<a href="https://colab.research.google.com/github/Svatunek-Lab/teaching/blob/main/2026_University_of_Malaya_Molecule_to_3D_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecule → 3D
Enter a **name, CAS number, or SMILES**. [Draw a molecule → copy SMILES](https://draw.cheminfo.org/).

In [ ]:
!pip -q install rdkit py3Dmol ipywidgets

In [ ]:

#@title Molecular Structure Generator { display-mode: "form" }

import html
import json
import base64
import urllib.parse
import urllib.request

import ipywidgets as widgets
from IPython.display import HTML, display, clear_output
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import py3Dmol

# --------------------------------------------------
# INPUTS
# --------------------------------------------------

molecule_input = widgets.Text(
    value="aspirin",
    placeholder="Name, CAS number, or SMILES",
    description="Molecule:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="510px")
)

charge_input = widgets.IntText(
    value=0,
    description="Charge:",
    layout=widgets.Layout(width="145px")
)

multiplicity_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=15,
    description="Multiplicity:",
    layout=widgets.Layout(width="190px")
)

generate_button = widgets.Button(
    description="Generate 3D",
    button_style="primary",
    icon="cube"
)

result_output = widgets.Output()


# --------------------------------------------------
# RESOLVE NAME / CAS / SMILES
# --------------------------------------------------

def resolve_smiles(text):
    text = text.strip()

    if not text:
        raise ValueError("Enter a name, CAS number, or SMILES.")

    # Interpret valid SMILES directly
    candidate = Chem.MolFromSmiles(text)
    if candidate is not None:
        return text

    # Otherwise use PubChem
    encoded = urllib.parse.quote(text, safe="")
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{encoded}/property/CanonicalSMILES/JSON"
    )

    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            data = json.load(response)

        item = data["PropertyTable"]["Properties"][0]

        for field in (
            "SMILES",
            "ConnectivitySMILES",
            "CanonicalSMILES",
            "IsomericSMILES"
        ):
            if item.get(field):
                return item[field]

    except Exception as exc:
        raise ValueError(
            f"Could not resolve {text!r} using PubChem."
        ) from exc

    raise ValueError("PubChem returned no SMILES.")


# --------------------------------------------------
# GENERATE 3D
# --------------------------------------------------

def generate_structure(smiles):
    base = Chem.MolFromSmiles(smiles)

    if base is None:
        raise ValueError("Invalid molecular structure.")

    mol = Chem.AddHs(base)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42

    if AllChem.EmbedMolecule(mol, params) != 0:
        raise ValueError("3D coordinate generation failed.")

    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMolecule(mol, maxIters=400)

    elif AllChem.UFFHasAllMoleculeParams(mol):
        AllChem.UFFOptimizeMolecule(mol, maxIters=400)

    return base, mol


# --------------------------------------------------
# FILE GENERATION
# --------------------------------------------------

def create_files(mol, label, charge, multiplicity):
    conf = mol.GetConformer()
    lines = []

    for atom in mol.GetAtoms():
        pos = conf.GetAtomPosition(atom.GetIdx())

        lines.append(
            f"{atom.GetSymbol():<2} "
            f"{pos.x:13.6f} "
            f"{pos.y:13.6f} "
            f"{pos.z:13.6f}"
        )

    coordinates = "\n".join(lines)

    xyz = (
        f"{len(lines)}\n"
        f"{label}\n"
        f"{coordinates}\n"
    )

    gaussian = (
        "%chk=molecule.chk\n"
        "#p HF/3-21G Opt\n\n"
        f"{label}\n\n"
        f"{charge} {multiplicity}\n"
        f"{coordinates}\n\n"
    )

    return xyz, gaussian


# --------------------------------------------------
# DOWNLOAD LINKS
# --------------------------------------------------

def download_link(content, filename, title):
    encoded = base64.b64encode(
        content.encode("utf-8")
    ).decode("ascii")

    return (
        f'<a download="{html.escape(filename)}" '
        f'href="data:text/plain;base64,{encoded}" '
        'style="'
        'display:inline-block;'
        'padding:9px 17px;'
        'margin-right:8px;'
        'background:#555;'
        'color:white;'
        'border-radius:4px;'
        'text-decoration:none;'
        'font-size:14px;'
        '">'
        f'{html.escape(title)}'
        '</a>'
    )


# --------------------------------------------------
# 3DMOL VIEWER: ISOLATED IFRAME
# --------------------------------------------------

def make_3d_viewer(mol, width=450, height=330):

    viewer = py3Dmol.view(width=width, height=height)

    viewer.addModel(Chem.MolToMolBlock(mol), "sdf")

    viewer.setStyle({
        "stick": {"radius": 0.17},
        "sphere": {"scale": 0.26}
    })

    viewer.setBackgroundColor("white")
    viewer.zoomTo()

    # Generate the complete py3Dmol HTML
    viewer_html = viewer._make_html()

    # Give the viewer its own positioning context.
    # The iframe isolates its CSS and JavaScript from Colab.
    iframe_document = f"""
    <!DOCTYPE html>
    <html>
    <head>
    <meta charset="utf-8">
    <style>
        html, body {{
            width: 100%;
            height: 100%;
            margin: 0;
            padding: 0;
            overflow: hidden;
        }}

        #viewer-container {{
            position: relative !important;
            width: {width}px !important;
            height: {height}px !important;
            overflow: hidden !important;
        }}

        #viewer-container canvas {{
            max-width: none;
        }}
    </style>
    </head>
    <body>
        <div id="viewer-container">
            {viewer_html}
        </div>
    </body>
    </html>
    """

    # Escape the iframe document for the srcdoc attribute
    escaped = html.escape(iframe_document, quote=True)

    return f"""
    <iframe
        srcdoc="{escaped}"
        width="{width}"
        height="{height}"
        scrolling="no"
        style="
            display:block;
            position:relative;
            width:{width}px;
            height:{height}px;
            border:0;
            overflow:hidden;
            background:white;
        "
    ></iframe>
    """


# --------------------------------------------------
# DISPLAY
# --------------------------------------------------

def render_molecule(_=None):

    generate_button.disabled = True

    try:
        query = molecule_input.value.strip()
        smiles = resolve_smiles(query)

        base, mol = generate_structure(smiles)

        # 2D depiction
        svg = Draw.MolsToGridImage(
            [base],
            molsPerRow=1,
            subImgSize=(390, 330),
            useSVG=True
        )

        picture = f"""
        <div style="
            position:relative;
            width:400px;
            height:340px;
            display:flex;
            align-items:center;
            justify-content:center;
            background:white;
        ">
            {str(svg)}
        </div>
        """

        # 3D depiction
        viewer_html = make_3d_viewer(mol)

        # Files
        xyz, gaussian = create_files(
            mol,
            query,
            charge_input.value,
            multiplicity_input.value
        )

        downloads = (
            download_link(xyz, "molecule.xyz", "Download XYZ")
            + download_link(gaussian, "molecule.com", "Download COM")
            + download_link(gaussian, "molecule.gjf", "Download GJF")
        )

        # Side-by-side display
        combined_html = f"""
        <div style="font-family:Arial,sans-serif;">

            <div style="margin:10px 0 14px;">
                <b>SMILES:</b>
                <code>{html.escape(smiles)}</code>
            </div>

            <div style="
                display:flex;
                gap:18px;
                flex-wrap:wrap;
                align-items:flex-start;
            ">

                <div style="width:400px;">
                    <b>2D structure</b>
                    {picture}
                </div>

                <div style="width:450px;">
                    <b>3D structure (rotatable)</b>
                    <div style="
                        position:relative;
                        width:450px;
                        height:330px;
                        overflow:hidden;
                        margin-top:8px;
                    ">
                        {viewer_html}
                    </div>
                </div>

            </div>

            <div style="margin:18px 0;">
                {downloads}
            </div>

        </div>
        """

        with result_output:
            clear_output(wait=True)
            display(HTML(combined_html))

            display(HTML(
                '<b>XYZ coordinates (Å)</b>'
            ))

            display(widgets.Textarea(
                value=xyz,
                layout=widgets.Layout(
                    width="100%",
                    height="230px"
                )
            ))

    except Exception as exc:
        with result_output:
            clear_output(wait=True)
            print("Error:", exc)

    finally:
        generate_button.disabled = False


# --------------------------------------------------
# INTERFACE
# --------------------------------------------------

generate_button.on_click(render_molecule)

display(widgets.VBox([
    widgets.HBox([
        molecule_input,
        generate_button
    ]),
    widgets.HBox([
        charge_input,
        multiplicity_input
    ]),
    result_output
]))

render_molecule()
